# 01 Tìm Hiểu Dữ Liệu

Notebook này xây dựng nền tảng dữ liệu cho cả ba phần của vòng thi: MCQ, EDA và Dự báo. Notebook tải mọi file CSV, phân tích cột ngày tháng, kiểm tra chất lượng dữ liệu cơ bản, xác nhận các giả định về khóa chính và khóa ngoại, xây dựng bản đồ kết nối bảng, và phân loại mục đích sử dụng của từng bảng theo nhiệm vụ.

## 1. Cài Đặt

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 160)

project_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "dataset").exists()), Path.cwd())
DATA_DIR = project_root / "dataset"
ARTIFACT_DIR = project_root / "artifacts" / "data_understanding"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Data directory:", DATA_DIR)
print("Artifact directory:", ARTIFACT_DIR)


Project root: C:\Users\admin\OneDrive - National Economics University\Documents\NCKH\DATATHON\Neu_BRT_Datathon
Data directory: C:\Users\admin\OneDrive - National Economics University\Documents\NCKH\DATATHON\Neu_BRT_Datathon\dataset
Artifact directory: C:\Users\admin\OneDrive - National Economics University\Documents\NCKH\DATATHON\Neu_BRT_Datathon\artifacts\data_understanding


## 2. Tải Tất Cả File CSV Và Phân Tích Cột Ngày Tháng

In [2]:
DATE_CANDIDATE_COLUMNS = [
    "date", "Date", "order_date", "ship_date", "delivery_date", "return_date",
    "review_date", "signup_date", "snapshot_date", "start_date", "end_date",
]

csv_paths = sorted(DATA_DIR.glob("*.csv"))

dfs = {}
load_summary = []

for path in csv_paths:
    table_name = path.stem
    header_cols = pd.read_csv(path, nrows=0).columns.tolist()
    parse_dates = [col for col in DATE_CANDIDATE_COLUMNS if col in header_cols]
    df = pd.read_csv(path, parse_dates=parse_dates, low_memory=False)
    dfs[table_name] = df
    load_summary.append({
        "table": table_name,
        "path": str(path.relative_to(project_root)),
        "rows": len(df),
        "columns": len(df.columns),
        "parsed_date_columns": ", ".join(parse_dates) if parse_dates else "-",
    })

load_summary_df = pd.DataFrame(load_summary).sort_values("table").reset_index(drop=True)
display(load_summary_df)


,table,path,rows,columns,parsed_date_columns
0,customers,dataset\customers.csv,121930,7,signup_date
1,geography,dataset\geography.csv,39948,4,-
2,inventory,dataset\inventory.csv,60247,17,snapshot_date
3,order_items,dataset\order_items.csv,714669,7,-
4,orders,dataset\orders.csv,646945,8,order_date
5,payments,dataset\payments.csv,646945,4,-
6,products,dataset\products.csv,2412,8,-
7,promotions,dataset\promotions.csv,50,10,"start_date, end_date"
8,returns,dataset\returns.csv,39939,7,return_date
9,reviews,dataset\reviews.csv,113551,7,review_date


## 3. Hình Dạng, Cột Và Các Hàng Mẫu

Với mỗi bảng, phần này in ra kích thước, danh sách cột và ba hàng đầu tiên để đối chiếu schema thực tế với mô tả bài toán.

In [3]:
overview_rows = []

for table_name, df in dfs.items():
    display(Markdown(f"### {table_name}"))
    print(f"Shape: {df.shape}")
    print(f"Columns ({len(df.columns)}): {list(df.columns)}")
    display(df.head(3))

    overview_rows.append({
        "table": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": ", ".join(df.columns),
    })

overview_df = pd.DataFrame(overview_rows).sort_values(["rows", "table"], ascending=[False, True]).reset_index(drop=True)
display(Markdown("### Dataset Summary"))
display(overview_df)


### customers

Shape: (121930, 7)
Columns (7): ['customer_id', 'zip', 'city', 'signup_date', 'gender', 'age_group', 'acquisition_channel']


,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search


### geography

Shape: (39948, 4)
Columns (4): ['zip', 'city', 'region', 'district']


...

### web_traffic

Shape: (3652, 7)
Columns (7): ['date', 'sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source']


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct


### Dataset Summary

,table,rows,columns,column_names
0,order_items,714669,7,"order_id, product_id, quantity, unit_price, discount_amount, promo_id, promo_id_2"
1,orders,646945,8,"order_id, order_date, customer_id, zip, order_status, payment_method, device_type, order_source"
2,payments,646945,4,"order_id, payment_method, payment_value, installments"
3,shipments,566067,4,"order_id, ship_date, delivery_date, shipping_fee"
4,customers,121930,7,"customer_id, zip, city, signup_date, gender, age_group, acquisition_channel"
5,reviews,113551,7,"review_id, order_id, product_id, customer_id, review_date, rating, review_title"
6,inventory,60247,17,"snapshot_date, product_id, stock_on_hand, units_received, units_sold, stockout_days, days_of_supply, fill_rate, stockout_flag, overstock..."
7,geography,39948,4,"zip, city, region, district"
8,returns,39939,7,"return_id, order_id, product_id, return_date, return_reason, return_quantity, refund_amount"
9,sales,3833,3,"Date, Revenue, COGS"


## 4. Ứng Viên Khóa Chính

Một cột được coi là ứng viên khóa chính khi nó trông giống như một định danh và hoàn toàn duy nhất trong bảng.

In [4]:
KEY_LIKE_COLUMNS = {"zip", "date", "Date"}

pk_rows = []
for table_name, df in dfs.items():
    for col in df.columns:
        is_key_like = col.endswith("_id") or col in KEY_LIKE_COLUMNS
        if not is_key_like:
            continue

        non_null = int(df[col].notna().sum())
        distinct = int(df[col].nunique(dropna=True))
        pk_rows.append({
            "table": table_name,
            "column": col,
            "row_count": len(df),
            "non_null_rows": non_null,
            "distinct_values": distinct,
            "missing_rows": len(df) - non_null,
            "uniqueness_ratio": round(distinct / len(df), 4) if len(df) else None,
            "candidate_primary_key": (non_null == len(df)) and (distinct == len(df)),
        })

pk_df = pd.DataFrame(pk_rows).sort_values(["candidate_primary_key", "table", "column"], ascending=[False, True, True]).reset_index(drop=True)

display(Markdown("### Candidate Primary Keys"))
display(pk_df[pk_df["candidate_primary_key"]].reset_index(drop=True))

display(Markdown("### Key-Like Columns That Are Not Primary Keys"))
display(pk_df[~pk_df["candidate_primary_key"]].reset_index(drop=True))


### Candidate Primary Keys

,table,column,row_count,non_null_rows,distinct_values,missing_rows,uniqueness_ratio,candidate_primary_key
0,customers,customer_id,121930,121930,121930,0,1.0,True
1,geography,zip,39948,39948,39948,0,1.0,True
2,orders,order_id,646945,646945,646945,0,1.0,True
3,payments,order_id,646945,646945,646945,0,1.0,True
4,products,product_id,2412,2412,2412,0,1.0,True
5,promotions,promo_id,50,50,50,0,1.0,True
6,returns,return_id,39939,39939,39939,0,1.0,True
7,reviews,review_id,113551,113551,113551,0,1.0,True
8,sales,Date,3833,3833,3833,0,1.0,True
9,sample_submission,Date,548,548,548,0,1.0,True


### Key-Like Columns That Are Not Primary Keys

,table,column,row_count,non_null_rows,distinct_values,missing_rows,uniqueness_ratio,candidate_primary_key
0,customers,zip,121930,121930,31491,0,0.2583,False
1,inventory,product_id,60247,60247,1624,0,0.0270,False
2,order_items,order_id,714669,714669,646945,0,0.9052,False
3,order_items,product_id,714669,714669,1598,0,0.0022,False
4,order_items,promo_id,714669,276316,50,438353,0.0001,False
5,orders,customer_id,646945,646945,90246,0,0.1395,False
6,orders,zip,646945,646945,29932,0,0.0463,False
7,returns,order_id,39939,39939,36062,0,0.9029,False
8,returns,product_id,39939,39939,1286,0,0.0322,False
9,reviews,customer_id,113551,113551,48676,0,0.4287,False


## 5. Kiểm Tra Khóa Ngoại Và Bản Đồ Kết Nối Bảng

Các mối quan hệ dưới đây tuân theo mô tả bài toán. Với mỗi mối quan hệ, notebook kiểm tra xem các giá trị khóa ngoại khác null có tồn tại trong bảng tham chiếu hay không, đồng thời suy ra nhãn cardinality đơn giản.

In [5]:
fk_checks = [
    ("customers", "zip", "geography", "zip"),
    ("orders", "customer_id", "customers", "customer_id"),
    ("orders", "zip", "geography", "zip"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "promo_id", "promotions", "promo_id"),
    ("order_items", "promo_id_2", "promotions", "promo_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("shipments", "order_id", "orders", "order_id"),
    ("returns", "order_id", "orders", "order_id"),
    ("returns", "product_id", "products", "product_id"),
    ("reviews", "order_id", "orders", "order_id"),
    ("reviews", "product_id", "products", "product_id"),
    ("reviews", "customer_id", "customers", "customer_id"),
    ("inventory", "product_id", "products", "product_id"),
]


def infer_cardinality(left_df, left_key, right_df, right_key):
    left_non_null = left_df[left_key].dropna()
    right_non_null = right_df[right_key].dropna()
    left_unique = left_non_null.is_unique
    right_unique = right_non_null.is_unique
    if left_unique and right_unique:
        return "1:1"
    if left_unique and not right_unique:
        return "1:n"
    if not left_unique and right_unique:
        return "n:1"
    return "n:n"


def profile_foreign_key(left_table, left_key, right_table, right_key):
    left_df = dfs[left_table]
    right_df = dfs[right_table]
    left_non_null = left_df[left_key].dropna()
    right_values = set(right_df[right_key].dropna().unique())
    unmatched_rows = int((~left_non_null.isin(right_values)).sum())
    checked_rows = int(len(left_df))
    non_null_left_rows = int(len(left_non_null))
    match_rate_pct = round((1 - unmatched_rows / non_null_left_rows) * 100, 2) if non_null_left_rows else None
    return {
        "left_table": left_table,
        "left_key": left_key,
        "right_table": right_table,
        "right_key": right_key,
        "checked_rows": checked_rows,
        "non_null_left_rows": non_null_left_rows,
        "unmatched_rows": unmatched_rows,
        "match_rate_pct": match_rate_pct,
        "relationship": infer_cardinality(left_df, left_key, right_df, right_key),
        "note": "OK" if unmatched_rows == 0 else "Has unmatched FK values",
    }

fk_df = pd.DataFrame([profile_foreign_key(*check) for check in fk_checks])
join_map_df = fk_df[["left_table", "left_key", "right_table", "right_key", "relationship", "note"]].copy()

display(Markdown("### Foreign-Key Validation"))
display(fk_df)

display(Markdown("### Relationships With Mismatches"))
display(fk_df[fk_df["unmatched_rows"] > 0].reset_index(drop=True))

display(Markdown("### Join Map"))
display(join_map_df)


### Foreign-Key Validation

,left_table,left_key,right_table,right_key,checked_rows,non_null_left_rows,unmatched_rows,match_rate_pct,relationship,note
0,customers,zip,geography,zip,121930,121930,0,100.0,n:1,OK
1,orders,customer_id,customers,customer_id,646945,646945,0,100.0,n:1,OK
2,orders,zip,geography,zip,646945,646945,0,100.0,n:1,OK
3,order_items,order_id,orders,order_id,714669,714669,0,100.0,n:1,OK
4,order_items,product_id,products,product_id,714669,714669,0,100.0,n:1,OK
5,order_items,promo_id,promotions,promo_id,714669,276316,0,100.0,n:1,OK
6,order_items,promo_id_2,promotions,promo_id,714669,206,0,100.0,n:1,OK
7,payments,order_id,orders,order_id,646945,646945,0,100.0,1:1,OK
8,shipments,order_id,orders,order_id,566067,566067,0,100.0,1:1,OK
9,returns,order_id,orders,order_id,39939,39939,0,100.0,n:1,OK


### Relationships With Mismatches

,left_table,left_key,right_table,right_key,checked_rows,non_null_left_rows,unmatched_rows,match_rate_pct,relationship,note


### Join Map

,left_table,left_key,right_table,right_key,relationship,note
0,customers,zip,geography,zip,n:1,OK
1,orders,customer_id,customers,customer_id,n:1,OK
2,orders,zip,geography,zip,n:1,OK
3,order_items,order_id,orders,order_id,n:1,OK
4,order_items,product_id,products,product_id,n:1,OK
5,order_items,promo_id,promotions,promo_id,n:1,OK
6,order_items,promo_id_2,promotions,promo_id,n:1,OK
7,payments,order_id,orders,order_id,1:1,OK
8,shipments,order_id,orders,order_id,1:1,OK
9,returns,order_id,orders,order_id,n:1,OK


## 6. Giá Trị Thiếu

In [6]:
missing_rows = []

for table_name, df in dfs.items():
    missing_count = df.isna().sum()
    missing_pct = (missing_count / len(df) * 100).round(2) if len(df) else missing_count
    for col in df.columns:
        if missing_count[col] > 0:
            missing_rows.append({
                "table": table_name,
                "column": col,
                "missing_count": int(missing_count[col]),
                "missing_pct": float(missing_pct[col]),
            })

missing_df = pd.DataFrame(missing_rows)
if missing_df.empty:
    missing_df = pd.DataFrame(columns=["table", "column", "missing_count", "missing_pct"])
else:
    missing_df = missing_df.sort_values(["missing_pct", "missing_count", "table", "column"], ascending=[False, False, True, True]).reset_index(drop=True)

display(Markdown("### Missing-Value Summary"))
display(missing_df)

display(Markdown("### Top Missing Columns"))
display(missing_df.head(15))


### Missing-Value Summary

,table,column,missing_count,missing_pct
0,order_items,promo_id_2,714463,99.97
1,promotions,applicable_category,40,80.00
2,order_items,promo_id,438353,61.34


### Top Missing Columns

,table,column,missing_count,missing_pct
0,order_items,promo_id_2,714463,99.97
1,promotions,applicable_category,40,80.00
2,order_items,promo_id,438353,61.34


## 7. Phạm Vi Ngày Tháng

In [7]:
date_rows = []

for table_name, df in dfs.items():
    date_cols = [col for col in df.columns if col in DATE_CANDIDATE_COLUMNS or pd.api.types.is_datetime64_any_dtype(df[col])]
    for col in date_cols:
        series = pd.to_datetime(df[col], errors="coerce").dropna()
        if series.empty:
            continue
        date_rows.append({
            "table": table_name,
            "date_column": col,
            "min_date": series.min().date(),
            "max_date": series.max().date(),
            "distinct_dates": int(series.nunique()),
        })

date_ranges_df = pd.DataFrame(date_rows).sort_values(["table", "date_column"]).reset_index(drop=True)
display(date_ranges_df)


,table,date_column,min_date,max_date,distinct_dates
0,customers,signup_date,2012-01-17,2022-12-31,3941
1,inventory,snapshot_date,2012-07-31,2022-12-31,126
2,orders,order_date,2012-07-04,2022-12-31,3833
3,promotions,end_date,2013-03-01,2022-12-31,50
4,promotions,start_date,2013-01-31,2022-11-18,50
5,returns,return_date,2012-07-11,2022-12-31,3806
6,reviews,review_date,2012-07-10,2022-12-31,3825
7,sales,Date,2012-07-04,2022-12-31,3833
8,sample_submission,Date,2023-01-01,2024-07-01,548
9,shipments,delivery_date,2012-07-06,2022-12-31,3831


## 8. Phân Loại Bảng Theo MCQ, EDA Và Dự Báo

Dự báo được hiểu đúng theo định nghĩa của bài toán: dự đoán `Revenue` và `COGS` trong chuỗi thời gian sales. Các bảng giao dịch và vận hành có thể hỗ trợ EDA hoặc feature engineering lịch sử, nhưng cần sử dụng thận trọng để tránh leakage khi tạo dự đoán cho giai đoạn kiểm tra.

In [8]:
use_case_rows = [
    {"table": "customers", "MCQ": True, "EDA": True, "Forecasting": "Indirect", "reason": "Customer segmentation by age group, gender, and acquisition channel; forecasting use only through valid historical aggregation."},
    {"table": "geography", "MCQ": True, "EDA": True, "Forecasting": "Indirect", "reason": "Reference table for city, district, and region joins."},
    {"table": "inventory", "MCQ": False, "EDA": True, "Forecasting": "Potential historical feature", "reason": "Monthly stock, stockout, fill-rate, and sell-through indicators can explain operational constraints."},
    {"table": "orders", "MCQ": True, "EDA": True, "Forecasting": "Potential historical feature", "reason": "Core transaction table with order dates, channels, devices, status, and geography."},
    {"table": "order_items", "MCQ": True, "EDA": True, "Forecasting": "Potential historical feature", "reason": "Line-level quantity, unit price, discount, product, and promotion usage."},
    {"table": "payments", "MCQ": True, "EDA": True, "Forecasting": "Indirect", "reason": "Payment method and installment mix support business analysis and MCQ answers."},
    {"table": "products", "MCQ": True, "EDA": True, "Forecasting": "Indirect", "reason": "Product dimension table for category, segment, size, price, and COGS."},
    {"table": "promotions", "MCQ": True, "EDA": True, "Forecasting": "Potential calendar feature", "reason": "Promotion metadata can support calendar-style features if known without leakage."},
    {"table": "returns", "MCQ": True, "EDA": True, "Forecasting": "Indirect", "reason": "Return reasons and return rates explain product quality and net revenue pressure."},
    {"table": "reviews", "MCQ": False, "EDA": True, "Forecasting": "Indirect", "reason": "Ratings and review titles support customer-satisfaction analysis."},
    {"table": "sales", "MCQ": False, "EDA": True, "Forecasting": "Target train", "reason": "Training target table with Date, Revenue, and COGS."},
    {"table": "sample_submission", "MCQ": False, "EDA": False, "Forecasting": "Submission format", "reason": "Required output schema and row order."},
    {"table": "submission", "MCQ": False, "EDA": False, "Forecasting": "Existing output", "reason": "Existing prediction file; it must not be used as a model feature."},
    {"table": "shipments", "MCQ": False, "EDA": True, "Forecasting": "Indirect", "reason": "Shipping delay and shipping-fee analysis for logistics insights."},
    {"table": "web_traffic", "MCQ": True, "EDA": True, "Forecasting": "Potential historical feature", "reason": "Daily traffic can act as a demand signal if only valid historical data is used."},
]

table_usage_df = pd.DataFrame(use_case_rows).sort_values("table").reset_index(drop=True)
display(table_usage_df)


,table,MCQ,EDA,Forecasting,reason
0,customers,True,True,Indirect,"Customer segmentation by age group, gender, and acquisition channel; forecasting use only through valid historical aggregation."
1,geography,True,True,Indirect,"Reference table for city, district, and region joins."
2,inventory,False,True,Potential historical feature,"Monthly stock, stockout, fill-rate, and sell-through indicators can explain operational constraints."
3,order_items,True,True,Potential historical feature,"Line-level quantity, unit price, discount, product, and promotion usage."
4,orders,True,True,Potential historical feature,"Core transaction table with order dates, channels, devices, status, and geography."
5,payments,True,True,Indirect,Payment method and installment mix support business analysis and MCQ answers.
6,products,True,True,Indirect,"Product dimension table for category, segment, size, price, and COGS."
7,promotions,True,True,Potential calendar feature,Promotion metadata can support calendar-style features if known without leakage.
8,returns,True,True,Indirect,Return reasons and return rates explain product quality and net revenue pressure.
9,reviews,False,True,Indirect,Ratings and review titles support customer-satisfaction analysis.


## 9. Xuất Các Artifact Kiểm Tra

In [9]:
exports = {
    "dataset_summary.csv": overview_df,
    "load_summary.csv": load_summary_df,
    "primary_key_profile.csv": pk_df,
    "foreign_key_profile.csv": fk_df,
    "join_map.csv": join_map_df,
    "missing_values.csv": missing_df,
    "date_ranges.csv": date_ranges_df,
    "table_usage.csv": table_usage_df,
}

for filename, df in exports.items():
    out_path = ARTIFACT_DIR / filename
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print("Saved", out_path.relative_to(project_root))


Saved artifacts\data_understanding\dataset_summary.csv
Saved artifacts\data_understanding\load_summary.csv
Saved artifacts\data_understanding\primary_key_profile.csv
Saved artifacts\data_understanding\foreign_key_profile.csv
Saved artifacts\data_understanding\join_map.csv
Saved artifacts\data_understanding\missing_values.csv
Saved artifacts\data_understanding\date_ranges.csv
Saved artifacts\data_understanding\table_usage.csv


## 10. Những Điểm Chính Rút Ra

In [10]:
candidate_pk_count = int(pk_df["candidate_primary_key"].sum()) if not pk_df.empty else 0
fk_mismatch_count = int((fk_df["unmatched_rows"] > 0).sum()) if not fk_df.empty else 0
tables_with_missing = int(missing_df["table"].nunique()) if not missing_df.empty else 0
tables_with_dates = int(date_ranges_df["table"].nunique()) if not date_ranges_df.empty else 0

summary_md = f"""
- Loaded **{len(dfs)}** CSV tables from `dataset/`.
- Found **{candidate_pk_count}** primary-key candidate columns using uniqueness checks.
- Found **{fk_mismatch_count}** foreign-key relationships with at least one unmatched value.
- Found missing values in **{tables_with_missing}** tables.
- Found date columns in **{tables_with_dates}** tables, enough for time-based EDA and forecasting preparation.
- Exported audit artifacts to `artifacts/data_understanding/`.
"""

display(Markdown(summary_md))



- Loaded **15** CSV tables from `dataset/`.
- Found **13** primary-key candidate columns using uniqueness checks.
- Found **0** foreign-key relationships with at least one unmatched value.
- Found missing values in **2** tables.
- Found date columns in **11** tables, enough for time-based EDA and forecasting preparation.
- Exported audit artifacts to `artifacts/data_understanding/`.
